In [2]:
# =========================================================
# [Cell 1] Imports
# =========================================================
import os
import json
import time
import random
from datetime import datetime, timezone
from typing import List, Dict, Any, Optional, Tuple

import requests
import pandas as pd

In [3]:
# =========================================================
# [Cell 2] CONFIG
# =========================================================

# ✅ 수집 대상 서브레딧 리스트 (원하는 만큼 추가 가능)
SUBREDDITS = [
    "leagueoflegends",
    "summonerschool",
    "LeagueOfMemes",
    "top_mains",
    "Jungle_Mains",
    "midlanemains",
    "ADCMains",
    "supportlol"
]

# Posts: top only, t=year only
POST_SORT = "top"
TOP_TIME_FILTER = "year"
TARGET_POSTS = 500

# Listing paging (100 * 5 = 500)
POSTS_LIMIT = 100
POSTS_PAGES = 5

# Comments
COLLECT_COMMENTS = True
COMMENT_SORT = "top"
COMMENTS_LIMIT = 500

# Rate limit 완화
SLEEP_BETWEEN_REQ = (0.1, 0.2)
SLEEP_BETWEEN_COMMENT_REQ = (0.1, 0.2)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) LastDanceLoLResearch/0.3"
}

# Output base
BASE_OUTDIR = "reddit_out"
os.makedirs(BASE_OUTDIR, exist_ok=True)

In [4]:
# =========================================================
# [Cell 3] Utils: sleep/state/http
# =========================================================

def jitter_sleep(a: float, b: float) -> None:
    time.sleep(random.uniform(a, b))

def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

def load_state(path: str) -> Dict[str, Any]:
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}

def save_state(path: str, obj: Dict[str, Any]) -> None:
    ensure_dir(os.path.dirname(path))
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False)

def safe_get_json(session: requests.Session, url: str, max_retries: int = 8) -> Optional[Any]:
    """
    - 무인증 reddit json은 429가 잦을 수 있음
    - Retry-After 있으면 따르고, 없으면 backoff
    - HTML로 떨어지는 경우(content-type)도 방어
    """
    backoff = 8
    for _ in range(max_retries):
        try:
            r = session.get(url, headers=HEADERS, timeout=30)
        except requests.RequestException:
            jitter_sleep(backoff, backoff + 1.5)
            backoff = min(backoff * 2, 180)
            continue

        if r.status_code == 200:
            ctype = (r.headers.get("content-type") or "").lower()
            if "application/json" not in ctype:
                jitter_sleep(backoff, backoff + 1.0)
                backoff = min(backoff * 2, 180)
                continue
            try:
                return r.json()
            except ValueError:
                return None

        if r.status_code == 429:
            ra = r.headers.get("Retry-After")
            wait = int(float(ra)) if ra else backoff
            time.sleep(wait + random.uniform(0, 1.0))
            print(f"429 received. Waited for {wait} seconds. Backoff now {backoff}.")
            backoff = min(backoff * 2, 180)
            continue

        # other errors
        jitter_sleep(backoff, backoff + 1.5)
        backoff = min(backoff * 2, 180)

    return None

In [5]:
# =========================================================
# [Cell 4] Parsers
# =========================================================

def parse_post(child: Dict[str, Any], subreddit: str) -> Dict[str, Any]:
    d = child.get("data", {})
    created = int(d.get("created_utc", 0))
    permalink = d.get("permalink", "")

    return {
        "subreddit": subreddit,
        "sort_source": POST_SORT,
        "time_filter": TOP_TIME_FILTER,

        "id": d.get("id", ""),
        "fullname": d.get("name", ""),  # t3_xxx

        "created_utc": created,
        "created_dt_utc": datetime.fromtimestamp(created, tz=timezone.utc).isoformat() if created else "",

        "title": d.get("title", ""),
        "score": int(d.get("score", 0)),
        "upvote_ratio": d.get("upvote_ratio", None),
        "num_comments": int(d.get("num_comments", 0)),

        "permalink": ("https://www.reddit.com" + permalink) if permalink else "",
        "selftext": d.get("selftext", ""),
        "url": d.get("url", ""),
        "is_self": bool(d.get("is_self", False)),
    }


def flatten_top_level_comments(children: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    out = []
    for ch in children:
        if ch.get("kind") != "t1":
            continue
        d = ch.get("data", {})
        body = d.get("body")
        if not body or body in ("[deleted]", "[removed]"):
            continue
        out.append({
            "comment_id": d.get("id"),
            "comment_fullname": d.get("name", ""),
            "comment_author": d.get("author"),
            "comment_score": int(d.get("score", 0)),
            "comment_created_utc": int(d.get("created_utc", 0)),
            "comment_body": body,
        })
    return out

In [6]:
# =========================================================
# [Cell 5] Step 1) Collect posts
# =========================================================

def fetch_posts_top_year(
    session: requests.Session,
    subreddit: str,
    pages: int = 5,
    limit: int = 100
) -> List[Dict[str, Any]]:
    base = (
        f"https://www.reddit.com/r/{subreddit}/{POST_SORT}.json"
        f"?t={TOP_TIME_FILTER}&limit={limit}&raw_json=1"
    )

    after = None
    rows: List[Dict[str, Any]] = []
    seen = set()

    for _ in range(pages):
        url = base + (f"&after={after}" if after else "")
        data = safe_get_json(session, url)
        if not data:
            break

        children = data.get("data", {}).get("children", [])
        if not children:
            break

        for ch in children:
            p = parse_post(ch, subreddit=subreddit)
            pid = p.get("id")
            if not pid or pid in seen:
                continue
            seen.add(pid)
            rows.append(p)

        after = data.get("data", {}).get("after")
        if not after:
            break

        jitter_sleep(*SLEEP_BETWEEN_REQ)

    return rows[:TARGET_POSTS]


def collect_posts_with_state(subreddit: str) -> pd.DataFrame:
    outdir = os.path.join(BASE_OUTDIR, f"reddit_out_{subreddit.lower()}")
    state_dir = os.path.join(outdir, "_state")
    ensure_dir(outdir)
    ensure_dir(state_dir)

    state_path = os.path.join(state_dir, "state_posts.json")
    st = load_state(state_path)

    if st.get("done") is True and isinstance(st.get("rows"), list) and len(st["rows"]) > 0:
        rows = st["rows"]
        print(f"[posts] loaded state r/{subreddit}: {len(rows)}")
    else:
        with requests.Session() as session:
            rows = fetch_posts_top_year(session, subreddit, pages=POSTS_PAGES, limit=POSTS_LIMIT)
        save_state(state_path, {"done": True, "rows": rows})
        print(f"[posts] fetched r/{subreddit}: {len(rows)}")

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    df = df.drop_duplicates("id").copy()
    df["created_dt"] = pd.to_datetime(df["created_utc"], unit="s", utc=True, errors="coerce")
    df = df.dropna(subset=["created_dt"]).copy()

    df = df.sort_values(["score", "num_comments", "created_dt"], ascending=[False, False, False])

    df = df.head(TARGET_POSTS).reset_index(drop=True)
    return df

In [7]:
# =========================================================
# [Cell 6] Step 2) Save posts
# =========================================================

def save_posts(subreddit: str, df_posts: pd.DataFrame) -> Tuple[str, str]:
    outdir = os.path.join(BASE_OUTDIR, f"reddit_out_{subreddit.lower()}")
    ensure_dir(outdir)

    posts_csv = os.path.join(outdir, f"posts_{subreddit}_{POST_SORT}_{TOP_TIME_FILTER}_{TARGET_POSTS}.csv")
    meta_json = os.path.join(outdir, f"meta_{subreddit}_{POST_SORT}_{TOP_TIME_FILTER}.json")

    if not df_posts.empty:
        df_posts.to_csv(posts_csv, index=False, encoding="utf-8-sig")

    meta = {
        "subreddit": subreddit,
        "post_sort": POST_SORT,
        "time_filter": TOP_TIME_FILTER,
        "target_posts": TARGET_POSTS,
        "collected_posts": int(len(df_posts)),
        "collected_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    with open(meta_json, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    print("[save] posts:", posts_csv, f"({len(df_posts)})")
    print("[save] meta :", meta_json)
    return posts_csv, meta_json

In [8]:
# =========================================================
# [Cell 7] Step 3) Collect comments
# =========================================================

def fetch_comments_for_post(
    session: requests.Session,
    permalink: str,
    limit: int = 100
) -> List[Dict[str, Any]]:
    if not permalink:
        return []

    url = permalink.rstrip("/") + f".json?limit={limit}&sort={COMMENT_SORT}&raw_json=1"
    data = safe_get_json(session, url)
    if not data or not isinstance(data, list) or len(data) < 2:
        return []

    children = data[1].get("data", {}).get("children", [])
    return flatten_top_level_comments(children)


def collect_comments_with_state(subreddit: str, df_posts: pd.DataFrame) -> pd.DataFrame:
    outdir = os.path.join(BASE_OUTDIR, f"reddit_out_{subreddit.lower()}")
    state_dir = os.path.join(outdir, "_state")
    ensure_dir(outdir)
    ensure_dir(state_dir)

    state_path = os.path.join(state_dir, "state_comments.json")
    st = load_state(state_path)

    done_ids = set(st.get("done_post_ids", []))
    rows = st.get("rows", [])

    total = len(df_posts)
    if total == 0:
        return pd.DataFrame()

    with requests.Session() as session:
        for idx, r in enumerate(df_posts.itertuples(index=False), 1):
            post_id = getattr(r, "id", "")
            if not post_id or post_id in done_ids:
                continue

            permalink = getattr(r, "permalink", "")
            comments = fetch_comments_for_post(session, permalink, limit=COMMENTS_LIMIT)

            for c in comments:
                rows.append({
                    "subreddit": subreddit,
                    "post_sort": POST_SORT,
                    "post_time_filter": TOP_TIME_FILTER,

                    "post_id": post_id,
                    "post_created_utc": int(getattr(r, "created_utc", 0)),
                    "post_title": getattr(r, "title", ""),
                    "post_score": int(getattr(r, "score", 0)),
                    "post_num_comments": int(getattr(r, "num_comments", 0)),
                    "post_permalink": permalink,

                    "comment_sort": COMMENT_SORT,
                    "comment_id": c.get("comment_id"),
                    "comment_fullname": c.get("comment_fullname"),
                    "comment_author": c.get("comment_author"),
                    "comment_score": int(c.get("comment_score", 0)),
                    "comment_created_utc": int(c.get("comment_created_utc", 0)),
                    "comment_body": c.get("comment_body", ""),
                })

            done_ids.add(post_id)
            save_state(state_path, {"done_post_ids": list(done_ids), "rows": rows})

            print(f"[comments] r/{subreddit} {idx}/{total} post_id={post_id} -> {len(comments)} top-level comments")
            jitter_sleep(*SLEEP_BETWEEN_COMMENT_REQ)

    dfc = pd.DataFrame(rows)
    if dfc.empty:
        return dfc

    dfc = dfc.drop_duplicates(["post_id", "comment_id"]).reset_index(drop=True)
    return dfc

In [9]:
# =========================================================
# [Cell 8] Step 4) Save comments
# =========================================================

def save_comments(subreddit: str, df_comments: pd.DataFrame) -> str:
    outdir = os.path.join(BASE_OUTDIR, f"reddit_out_{subreddit.lower()}")
    ensure_dir(outdir)

    comments_csv = os.path.join(
        outdir,
        f"comments_{subreddit}_from_{TARGET_POSTS}posts_{COMMENT_SORT}_limit{COMMENTS_LIMIT}.csv"
    )
    if not df_comments.empty:
        df_comments.to_csv(comments_csv, index=False, encoding="utf-8-sig")

    print("[save] comments:", comments_csv, f"({len(df_comments)})")
    return comments_csv

In [10]:
# =========================================================
# [Cell 9] RUN
# =========================================================

for sub in SUBREDDITS:
    print("\n" + "="*90)
    print(f"SUBREDDIT: r/{sub} | posts={TARGET_POSTS} | sort=top | t=year | comments<= {COMMENTS_LIMIT}/post")
    print("="*90)

    # Posts
    df_posts = collect_posts_with_state(sub)
    posts_csv, meta_json = save_posts(sub, df_posts)

    print("\n[posts summary]")
    print(" - collected:", len(df_posts))
    if len(df_posts) > 0:
        print(" - newest  :", df_posts["created_dt"].max())
        print(" - oldest  :", df_posts["created_dt"].min())

    # Comments
    if COLLECT_COMMENTS:
        print("\nCollecting comments...")
        df_comments = collect_comments_with_state(sub, df_posts)
        comments_csv = save_comments(sub, df_comments)

        print("\n[comments summary]")
        print(" - total rows:", len(df_comments))
        if len(df_comments) > 0:
            n_posts = df_comments["post_id"].nunique()
            print(" - posts with >=1 comment row:", n_posts)
            print(" - avg comment rows/post    :", round(len(df_comments) / max(n_posts, 1), 2))

print("\n==== DONE ====")
print("BASE_OUTDIR:", BASE_OUTDIR)


SUBREDDIT: r/leagueoflegends | posts=500 | sort=top | t=year | comments<= 500/post
[posts] fetched r/leagueoflegends: 498
[save] posts: reddit_out\reddit_out_leagueoflegends\posts_leagueoflegends_top_year_500.csv (498)
[save] meta : reddit_out\reddit_out_leagueoflegends\meta_leagueoflegends_top_year.json

[posts summary]
 - collected: 498
 - newest  : 2026-02-16 20:33:03+00:00
 - oldest  : 2025-02-17 21:43:23+00:00

[comments] r/leagueoflegends 1/498 post_id=1oqrh3s -> 195 top-level comments
[comments] r/leagueoflegends 2/498 post_id=1itvd1a -> 257 top-level comments
[comments] r/leagueoflegends 3/498 post_id=1q35gra -> 81 top-level comments
[comments] r/leagueoflegends 4/498 post_id=1oshrk9 -> 107 top-level comments
[comments] r/leagueoflegends 5/498 post_id=1ozegty -> 105 top-level comments
[comments] r/leagueoflegends 6/498 post_id=1qtaxmi -> 111 top-level comments
[comments] r/leagueoflegends 7/498 post_id=1irvs5p -> 86 top-level comments
[comments] r/leagueoflegends 8/498 post_id